In [ ]:
import sqlite3
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import seaborn as sns
from IPython.display import display

sys.path.append("..")

from src.data_processing import execute_query, statistical_summary
from src.utils import (
    format_code_blocks,
    format_dict,
    format_section,
    format_table,
    read_file_content,
)

## Esquema de Tablas en la Base de Datos

El código consulta el esquema de la base de datos SQLite a partir de la tabla interna `sqlite_master`. Para ello, se carga desde un archivo externo la consulta SQL correspondiente, se establece una conexión con `cobranza.db` y se ejecuta la consulta mediante la función `execute_query()`. Finalmente, `format_code_blocks()` permite visualizar los resultados como bloques de código SQL dentro del notebook, usando el nombre de cada tabla como título del bloque.

La salida permite identificar que la base de datos está compuesta por dos tablas principales: `clientes` e `historial_mensual`. La tabla `clientes` contiene información relativamente estable de cada persona, como edad, sexo, educación, estado civil, límite de crédito y la variable `default_pago`. Por otro lado, `historial_mensual` contiene información que cambia a través del tiempo, como el estado de pago, el monto facturado y el monto pagado para cada cliente y mes.

Desde una perspectiva de Data Science, esta separación es importante porque permite distinguir entre **variables estáticas o demográficas** y **variables temporales o de comportamiento**. Para un modelo de Machine Learning orientado a predecir el comportamiento de pago, ambas fuentes pueden aportar información relevante: las características del cliente pueden representar su perfil, mientras que el historial mensual puede proporcionar señales sobre su comportamiento financiero previo.

También se puede observar que `cliente_id` aparece en ambas tablas, por lo que funciona como la clave utilizada para relacionar la información de un cliente con su historial. Sin embargo, el esquema no define explícitamente una `PRIMARY KEY` ni restricciones `FOREIGN KEY`. Esto no impide realizar los análisis, pero significa que la integridad de las relaciones debe comprobarse mediante exploración de los datos. Por ejemplo, será necesario verificar si existen clientes duplicados, registros de historial sin cliente correspondiente o más de un registro para un mismo cliente y mes.

La variable `default_pago` merece especial atención porque será una posible **variable objetivo (target)** del proyecto. Antes de utilizarla en un modelo, es necesario comprender exactamente qué representa, qué valores puede tomar y cuál es su distribución. Si representa si un cliente incumple o no sus pagos, podría plantearse como un problema de **clasificación binaria**, siempre que tenga únicamente dos categorías relevantes.

Finalmente, conocer el esquema antes de comenzar el análisis evita trabajar con el dataset como si fuera una simple tabla sin estructura. En un proyecto real de Machine Learning, una etapa fundamental consiste en entender de dónde provienen los datos, cómo están relacionados y qué significa cada variable antes de realizar limpieza, crear características o entrenar un modelo. Este paso constituye, por tanto, una primera revisión de la estructura y del contexto de los datos que posteriormente serán utilizados en el análisis exploratorio y en la construcción del modelo.

In [ ]:
format_section(
    title="Esquema de Tablas en la Base de Datos",
    subtitle="Consulta de metadatos y código SQL DDL de la base de datos sqlite_master"
)

query_tables = read_file_content("../queries/query_tables.sql")

conn = sqlite3.connect("../data/external/cobranza.db")
cursor = conn.cursor()
tables = execute_query(cursor, query_tables)

format_code_blocks(
    [[sql] for _, sql in tables],
    language="sql",
    titles=[name for name, _ in tables],
    render=True
)

## Exploración Inicial de la Estructura y los Registros

En esta sección se realiza una exploración más detallada de las tablas identificadas anteriormente. El código utiliza `PRAGMA table_info()` para consultar los metadatos de cada tabla y posteriormente ejecuta consultas exploratorias sobre la cantidad de registros por cliente, los primeros registros de `clientes` y el historial mensual. Los resultados se presentan mediante tablas con formato HTML para facilitar su lectura dentro del notebook.

La información obtenida mediante `PRAGMA` permite confirmar el nombre y tipo de cada columna. En `clientes`, las variables demográficas y financieras utilizan tipos `INTEGER` y `REAL`, mientras que `default_pago` también está almacenada como un entero. En `historial_mensual` se encuentran las variables relacionadas con el comportamiento mensual de pago. Un aspecto importante es que todas las columnas aparecen con `notnull = 0` y `pk = 0`. Esto indica que la base de datos no establece restricciones de valores nulos ni una clave primaria explícita. Por esta razón, la calidad y consistencia de estas variables deberán comprobarse durante la etapa de análisis y limpieza.

La consulta sobre la cantidad de registros muestra que los primeros cinco clientes tienen exactamente seis registros en `historial_mensual`. Esto es consistente con la estructura temporal de la base de datos, donde cada cliente parece contar con información correspondiente a seis meses. Esta característica será importante posteriormente porque permite analizar la evolución del comportamiento de pago de cada cliente y construir variables basadas en su historial.

La exploración de `clientes` permite observar cómo se representan algunas de las variables. Por ejemplo, `sexo`, `educacion` y `estado_civil` están codificadas numéricamente. Sin embargo, que una variable esté almacenada como `INTEGER` no significa necesariamente que sea una variable numérica continua. Estas variables pueden representar categorías y, antes de utilizarlas en Machine Learning, será necesario conocer el significado de cada código y decidir una representación adecuada para el modelo.

El historial mensual muestra una estructura temporal en la que cada cliente posee información de `estado_pago`, `monto_facturado` y `monto_pagado para` cada uno de los seis meses. Esto permite identificar patrones de comportamiento. Por ejemplo, en el cliente 1 se observa que durante los primeros meses no existe un monto facturado y posteriormente aparecen valores pendientes, mientras que otros clientes presentan pagos parciales o diferentes estados de pago. Estas variaciones pueden contener información predictiva relevante para determinar el riesgo de incumplimiento.

También aparecen valores como `-2`, `-1` y `0` en `estado_pago`. Antes de realizar transformaciones o entrenar un modelo, es fundamental investigar qué representa cada código. No sería correcto asumir automáticamente que `-1` o `-2` son valores inválidos o datos faltantes. En datasets financieros es frecuente que determinados valores numéricos sean códigos que representan estados específicos. La documentación de la fuente y la exploración de los datos permitirán determinar si deben mantenerse, transformarse o recodificarse.

Desde la perspectiva de Machine Learning, esta exploración ayuda a identificar una decisión importante: **qué información estará disponible en el momento en que se realice una predicción**. El historial mensual puede convertirse en características como pagos acumulados, proporción pagada respecto al monto facturado, número de meses con atraso o evolución del estado de pago. Sin embargo, estas características deben construirse utilizando únicamente información disponible antes del evento que se intenta predecir, para evitar *data leakage*.

En conjunto, esta exploración confirma que la base contiene dos niveles de información: características asociadas al cliente y un historial temporal de comportamiento. El siguiente paso será estudiar la calidad de estos datos —valores nulos, duplicados, rangos, categorías y posibles inconsistencias— antes de construir el dataset final que será utilizado para el análisis exploratorio y posteriormente para el entrenamiento del modelo.

In [ ]:
header_pragma_table = ["cid", "name", "type", "notnull", "dflt_value", "pk"]

format_section("PRAGMA Table Info: Clientes")
clientes_info = execute_query(cursor, "PRAGMA table_info(clientes);")
format_table(header_pragma_table, clientes_info, render=True)

format_section("PRAGMA Table Info: Historial Mensual")
historial_info = execute_query(cursor, "PRAGMA table_info(historial_mensual);")
format_table(header_pragma_table, historial_info, render=True)

format_section("Exploratory Registres Number")
query_registres = read_file_content("../queries/query_exploratory_registres_number.sql")
data_registres = execute_query(cursor, query_registres)
format_table(["cliente_id", "cantidad_registros"], data_registres, render=True)

format_section("Exploratory Clientes")
query_clientes = read_file_content("../queries/query_exploratory_clientes.sql")
data_clientes = execute_query(cursor, query_clientes)
headers_clientes = ["cliente_id", "edad", "sexo", "educacion", "estado_civil", "limite_credito", "default_pago"]
format_table(headers_clientes, data_clientes, render=True)

format_section("Exploratory Historial Mensual")
query_historial = read_file_content("../queries/query_exploratory_historial_mensual.sql")
data_historial = execute_query(cursor, query_historial)
encabezado_historial = [
    "cliente_id",
    "mes_1", "estado_pago_mes_1", "monto_facturado_mes_1", "pago_mes_1",
    "mes_2", "estado_pago_mes_2", "monto_facturado_mes_2", "pago_mes_2",
    "mes_3", "estado_pago_mes_3", "monto_facturado_mes_3", "pago_mes_3",
    "mes_4", "estado_pago_mes_4", "monto_facturado_mes_4", "pago_mes_4",
    "mes_5", "estado_pago_mes_5", "monto_facturado_mes_5", "pago_mes_5",
    "mes_6", "estado_pago_mes_6", "monto_facturado_mes_6", "pago_mes_6"
]
format_table(encabezado_historial, data_historial, render=True)

## Documentación de Variables

Para interpretar correctamente los resultados del análisis exploratorio es necesario conocer el significado de cada variable y distinguir entre variables numéricas, categóricas y ordinales. La siguiente tabla relaciona las variables originales del dataset de crédito con los nombres utilizados en el DataFrame del proyecto.

| Variable original              | Variable en el DataFrame                          | Descripción                                    | Categorías / Valores                                                                                                                   |
| ------------------------------ | ------------------------------------------------- | ---------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------- |
| **ID**                         | `cliente_id`                                      | Identificador del cliente                      | Entero                                                                                                                                 |
| **LIMIT_BAL**                  | `limite_credito`                                  | Monto del crédito otorgado                     | Numérico continuo                                                                                                                      |
| **SEX**                        | `sexo`                                            | Sexo del cliente                               | 1 = Masculino, 2 = Femenino                                                                                                            |
| **EDUCATION**                  | `educacion`                                       | Nivel educativo alcanzado                      | 1 = Posgrado, 2 = Universidad, 3 = Preparatoria, 4 = Otros, 5 = Desconocido, 6 = Desconocido                                           |
| **MARRIAGE**                   | `estado_civil`                                    | Estado civil                                   | 1 = Casado, 2 = Soltero, 3 = Otros                                                                                                     |
| **AGE**                        | `edad`                                            | Edad del cliente                               | Entero                                                                                                                                 |
| **PAY_0 a PAY_6**              | `estado_pago_mes_1` a `estado_pago_mes_6`         | Estado de pago mensual                         | -2 = sin consumo, -1 = pago puntual, 0 = crédito revolvente, 1 = retraso de 1 mes, 2 = retraso de 2 meses, ..., 8 = retraso de 8 meses |
| **BILL_AMT1 a BILL_AMT6**      | `monto_facturado_mes_1` a `monto_facturado_mes_6` | Monto facturado mensualmente                   | Numérico continuo                                                                                                                      |
| **PAY_AMT1 a PAY_AMT6**        | `pago_mes_1` a `pago_mes_6`                       | Monto pagado en cada período                   | Numérico continuo                                                                                                                      |
| **default.payment.next.month** | `default_pago`                                    | Incumplimiento de pago en el período siguiente | 1 = Sí, 0 = No                                                                                                                         |

> **Nota:** Los códigos de `PAY_0` a `PAY_6` no deben interpretarse simplemente como números continuos. Algunos valores representan estados especiales, como ausencia de consumo o pago puntual. Por ello, su significado debe conservarse durante la limpieza y transformación de características.

## Diagnóstico de Calidad de Datos

El diagnóstico inicial muestra que el dataset contiene **30.000 registros sin valores nulos y sin filas completamente duplicadas**. Esto es un resultado favorable desde el punto de vista de calidad básica, ya que no será necesario realizar inicialmente técnicas de imputación para valores `NaN`. Sin embargo, la ausencia de valores nulos y duplicados no significa que el dataset esté completamente limpio. Todavía es necesario investigar valores fuera de rango, códigos especiales, inconsistencias y posibles valores extremos.

La variable `edad` presenta una media de aproximadamente 35,5 años, una mediana de 34 y un rango entre 21 y 79 años. El método del IQR identifica 272 observaciones como posibles valores atípicos debido a que superan el límite superior de 60,5 años. Estos registros no deben eliminarse automáticamente. Una edad de 61, 70 o incluso 79 años puede ser perfectamente válida; en este caso, el resultado del IQR indica que son observaciones poco frecuentes respecto a la distribución central, no necesariamente errores. Esta distinción es fundamental en Data Science: **un outlier estadístico no es necesariamente un dato incorrecto**.

`limite_credito` presenta una dispersión considerable, con una mediana de 140.000 y valores entre 10.000 y 1.000.000. El IQR identifica 167 posibles outliers por encima de 525.000. Nuevamente, estos valores deben investigarse antes de eliminarlos, porque pueden representar clientes con una capacidad crediticia legítimamente superior al promedio. Además, esta variable tiene una escala mucho mayor que algunas otras características, algo que posteriormente deberá considerarse al seleccionar algoritmos y técnicas de escalamiento.

El comportamiento más llamativo aparece en las variables `monto_facturado_mes_X`. Todas presentan una gran cantidad de valores detectados como outliers mediante IQR. Por ejemplo, `monto_facturado_mes_1` tiene 2.693 observaciones clasificadas como atípicas y un máximo de 961.664, mientras que `monto_facturado_mes_4` alcanza un máximo de 1.664.089 y presenta 2.469 observaciones consideradas outliers. Además, existen valores negativos en los montos facturados. Esto constituye una **anomalía que debe investigarse específicamente**, porque un monto facturado negativo podría representar un ajuste, devolución, crédito o error de los datos. No es correcto decidir que todos los valores negativos deben eliminarse sin conocer primero su significado.

Las variables `pago_mes_X` también presentan una cantidad elevada de posibles outliers. Por ejemplo, `pago_mes_1` tiene 2.958 observaciones por encima del límite superior calculado mediante IQR y alcanza un máximo de 528.666. Este comportamiento puede explicarse parcialmente por una distribución muy asimétrica: la mayoría de los pagos son relativamente pequeños, mientras que un grupo reducido de clientes realiza pagos considerablemente mayores. Por tanto, estos valores requieren análisis de distribución y contexto financiero antes de aplicar cualquier transformación.

Las variables categóricas muestran comportamientos diferentes. `sexo` contiene únicamente las categorías 1 y 2, mientras que `estado_civil` contiene los valores 0, 1, 2 y 3. En este último caso aparece el valor 0, que no está descrito explícitamente en la tabla original proporcionada, por lo que debe investigarse antes de decidir si representa una categoría válida o un valor desconocido. `educacion` contiene siete valores distintos: 0, 1, 2, 3, 4, 5 y 6. Los valores 4, 5 y 6 aparecen con frecuencias mucho menores que las categorías principales. Esto es relevante porque podría existir una codificación de categorías desconocidas o poco frecuentes que posteriormente deba agruparse.

Es importante notar que el cálculo de outliers mediante IQR **no es apropiado para interpretar directamente variables categóricas**. Por ejemplo, `default_pago` presenta 6.636 valores identificados como "outliers" porque su IQR es cero y los valores `1` se encuentran fuera del intervalo calculado. Esto no significa que los 6.636 casos sean errores. Al contrario, representan exactamente la clase positiva del target. En una variable binaria, el concepto de outlier estadístico basado en IQR no tiene el mismo significado que en una variable continua. Por esta razón, el resultado debe ignorarse para determinar calidad de `default_pago`.

Algo similar ocurre con las variables `estado_pago_mes_X`. El método IQR identifica miles de observaciones como posibles outliers porque la mayor parte de los registros se concentra en valores como `-1`, `0` y `2`. Sin embargo, valores como `3`, `4`, `5` u `8` pueden representar diferentes niveles de retraso y, por tanto, son **categorías válidas con significado de negocio**, no necesariamente anomalías. Esto demuestra una limitación importante de aplicar automáticamente una misma técnica estadística a todos los tipos de variables.

Desde la perspectiva de Machine Learning, esta exploración permite establecer una primera clasificación de los problemas que deben investigarse. Los valores nulos y duplicados no representan actualmente un problema, pero existen posibles anomalías en los montos negativos, valores extremos de facturación y pagos, categorías poco frecuentes y códigos especiales de las variables de estado de pago. Antes de eliminar cualquier registro será necesario determinar si se trata de errores, valores legítimos o categorías con significado específico.

También es importante diferenciar entre **detección de anomalías y tratamiento de anomalías**. En esta etapa solamente estamos identificando observaciones que requieren investigación. El hecho de que el IQR marque un registro como outlier no significa que debamos eliminarlo. En un problema de riesgo crediticio, eliminar precisamente los clientes con comportamientos financieros extremos podría eliminar información predictiva importante y producir un modelo menos representativo de los casos reales.

Finalmente, el análisis confirma que `default_pago` es una variable binaria adecuada como candidato a **target de clasificación**. Existen 23.364 clientes con valor `0` y 6.636 con valor `1`, equivalente aproximadamente a un 22,1 % de incumplimiento. Esto representa un desbalance moderado de clases que deberá considerarse durante la evaluación del modelo. Métricas como *precision*, *recall*, *F1-score* y especialmente la matriz de confusión serán más informativas que utilizar únicamente *accuracy*.

En conclusión, el dataset presenta una buena calidad estructural inicial, pero todavía contiene varios aspectos que requieren investigación. Las principales prioridades para la siguiente etapa son **comprender los códigos de las variables categóricas y ordinales, investigar los valores negativos de los montos facturados, analizar los valores extremos de pagos y facturación y decidir cómo representar estas variables para Machine Learning**. La limpieza debe realizarse a partir del significado de los datos y del problema de negocio, no únicamente a partir de reglas estadísticas automáticas.

In [ ]:
format_section("Diagnóstico de Calidad de Datos")
quality_metrics = {
    "Total Valores Nulos (NaN)": int(df.isnull().sum().sum()),
    "Total Filas Duplicadas": int(df.duplicated().sum())
}
format_dict(quality_metrics, headers=("Métrica", "Cantidad"), render=True)

variables_numericas = [
    "edad",
    "limite_credito",
    "monto_facturado_mes_1",
    "monto_facturado_mes_2",
    "monto_facturado_mes_3",
    "monto_facturado_mes_4",
    "monto_facturado_mes_5",
    "monto_facturado_mes_6",
    "pago_mes_1",
    "pago_mes_2",
    "pago_mes_3",
    "pago_mes_4",
    "pago_mes_5",
    "pago_mes_6"
]

variables_categoricas = [
    "sexo",
    "educacion",
    "estado_civil",
    "default_pago"
]

variables_ordinales = [
    "estado_pago_mes_1",
    "estado_pago_mes_2",
    "estado_pago_mes_3",
    "estado_pago_mes_4",
    "estado_pago_mes_5",
    "estado_pago_mes_6"
]

format_section("Análisis de Variables Numéricas")
for variable in variables_numericas:
    summary = statistical_summary(df, variable)
    format_section(f"Variable Numérica: {variable}")
    format_dict(summary, headers=("Métrica", "Valor"), render=True)

format_section("Análisis de Variables Categóricas")
for variable in variables_categoricas:
    summary = statistical_summary(df, variable)
    format_section(f"Variable Categórica: {variable}")
    format_dict(summary, headers=("Métrica", "Valor"), render=True)

format_section("Análisis de Variables Ordinales")
for variable in variables_ordinales:
    summary = statistical_summary(df, variable)
    format_section(f"Variable Ordinal: {variable}")
    format_dict(summary, headers=("Métrica", "Valor"), render=True)

| Variable                      | Valores documentados                                                                                     | Valores no documentados / por investigar | Conclusión                                                                      |
| ----------------------------- | -------------------------------------------------------------------------------------------------------- | ---------------------------------------- | ------------------------------------------------------------------------------- |
| `educacion`                   | `1` = Posgrado, `2` = Universidad, `3` = Preparatoria, `4` = Otros, `5` = Desconocido, `6` = Desconocido | `0`                                      | El valor `0` no está documentado y debe investigarse.                           |
| `estado_civil`                | `1` = Casado, `2` = Soltero, `3` = Otros                                                                 | `0`                                      | El valor `0` no está documentado y debe investigarse.                           |
| `estado_pago_mes_1` a `6`     | `-1` = Pago puntual; `1` = retraso 1 mes; `2` = retraso 2 meses; ... `8` = retraso 8 meses               | `-2` y `0` según la tabla presentada     | `-2` y `0` requieren confirmación documental antes de transformarlos.           |
| `monto_facturado_mes_1` a `6` | Variable continua; monto facturado                                                                       | Valores negativos                        | Los negativos requieren investigación: podrían ser ajustes, créditos o errores. |
| `pago_mes_1` a `6`            | Variable continua; monto pagado                                                                          | Ningún código especial documentado       | Los valores extremos requieren análisis, pero pueden ser pagos legítimos.       |

## Investigación de Valores No Documentados y Anomalías Específicas

Este bloque de código profundiza en los hallazgos señalados en el diagnóstico de calidad anterior, examinando de forma puntual cada uno de los valores y patrones que quedaron marcados como "por investigar". A diferencia del análisis estadístico general realizado previamente, aquí se filtran subconjuntos específicos del DataFrame —mediante `df[condición]`— para observar directamente los registros que contienen los valores no documentados en `educacion` y `estado_civil`, los códigos `-2` y `0` en las seis variables `estado_pago_mes_X`, los montos facturados negativos en cada mes, los casos donde el pago excede al monto facturado del mismo período, y finalmente los clientes con `limite_credito` en el percentil 95 o superior. En cada caso se ordenan los resultados por `cliente_id` de forma descendente y se muestran las primeras diez filas, lo que permite una inspección cualitativa rápida sin necesidad de recorrer las 30.000 observaciones completas.

Respecto a `educacion = 0` y `estado_civil = 0`, la exploración confirma que estos registros no presentan ningún otro patrón obviamente anómalo: las edades, límites de crédito y variables de pago se comportan de manera similar al resto del dataset. Esto sugiere que el valor `0` probablemente funciona como una categoría residual de tipo "no informado" o "no clasificado", de forma análoga a como `educacion` ya incluye categorías igualmente ambiguas como `4`, `5` y `6`. Antes de imputar o recodificar, sería razonable agrupar estos valores minoritarios y poco frecuentes en una única categoría "Otros/Desconocido", una práctica común cuando una variable categórica tiene niveles con muy pocas observaciones, ya que evita que el modelo intente aprender patrones a partir de muestras estadísticamente insignificantes.

El análisis de `estado_pago_mes_X = -2` y `= 0` es particularmente revelador. Se observa que los clientes con `-2` en un mes determinado tienden a repetir ese mismo valor en varios meses consecutivos, y sus montos facturados y pagados suelen ser cero o muy bajos. Esto es consistente con la hipótesis de que `-2` representa "sin consumo" o "cuenta sin actividad", tal como se documentó posteriormente en la tabla de referencia. Por su parte, los registros con `estado_pago = 0` muestran montos facturados sustanciales y montos pagados también relevantes, lo cual respalda la interpretación de que `0` corresponde a "pago mínimo cumplido" o "uso de crédito revolvente sin retraso", es decir, un cliente que consume su cupo y realiza al menos el pago mínimo exigido, sin llegar a clasificarse como atrasado. Esta distinción es importante desde la perspectiva de *feature engineering*: tratar `-2`, `-1` y `0` como si fueran simplemente puntos intermedios en una escala numérica de "meses de atraso" sería un error conceptual, porque estos tres valores representan estados cualitativamente distintos (sin consumo, pago puntual, y uso normal del revolvente) que anteceden a los verdaderos niveles de atraso (`1` en adelante).

En cuanto a los montos facturados negativos, la inspección de los registros muestra magnitudes variadas: algunos casos son diferencias muy pequeñas (por ejemplo, `-3`, `-8`, `-18`) que podrían deberse a ajustes contables o redondeos, mientras que otros presentan montos considerablemente más grandes, como `-94.625` o `-11.798`. Este último tipo de valor es más difícil de explicar como un simple ajuste menor y refuerza la necesidad de consultar la documentación original del dataset o, en un entorno de producción real, al equipo de negocio que administra los datos. Una hipótesis razonable, común en datasets de tarjetas de crédito, es que un monto facturado negativo represente un saldo a favor del cliente (por ejemplo, tras un pago en exceso o una devolución), en cuyo caso no correspondería a un error sino a un estado financiero legítimo que podría incluso ser una señal predictiva relevante del comportamiento del cliente.

El análisis de los casos donde `pago_mes_X > monto_facturado_mes_X` complementa esta hipótesis: se identifican numerosos clientes que pagan más de lo que se les facturó en ese mes, en ocasiones por montos muy superiores (como el cliente 29964, cuyo pago del mes 4 casi duplica el monto facturado). Esto es consistente con clientes que liquidan saldos pendientes de meses anteriores, pagan por adelantado, o corrigen deudas acumuladas. Desde el punto de vista de modelado, esta relación entre lo facturado y lo pagado en cada periodo podría convertirse en una variable derivada útil —por ejemplo, una razón de pago sobre facturación— que capture el comportamiento de sobrepago como una señal de buen manejo financiero, potencialmente correlacionada con una menor probabilidad de incumplimiento.

Finalmente, la exploración de los valores extremos en `limite_credito` (percentil 95 o superior, con 1.558 registros) no revela ningún patrón sistemático de error: los clientes con límites altos presentan variables de edad, educación y comportamiento de pago dentro de rangos razonables, lo que refuerza la conclusión de la sección anterior de que estos valores corresponden a clientes con mayor capacidad crediticia y no a errores de captura. Esto ejemplifica un principio central en la preparación de datos para Machine Learning: el criterio estadístico (como el IQR o los percentiles) únicamente señala *dónde mirar*, pero la decisión final sobre si un valor es válido, debe transformarse o debe tratarse como un caso especial depende del contexto del negocio y de la evidencia acumulada en la exploración cualitativa, no de una regla automática aplicada de forma ciega. En conjunto, estos hallazgos consolidan la base necesaria para definir, en la siguiente etapa del proyecto, las reglas de limpieza y recodificación que se aplicarán antes de construir las características del modelo.

In [ ]:
format_section(
    title="Análisis de Variable educacion no documentada",
    subtitle="Educacion = 0"
)

display(df[df["educacion"] == 0].sort_values("cliente_id", ascending=False).head(5))

format_section(
    title="Análisis de Variable estado civil no documentada",
    subtitle="Estado civil = 0"
)

display(df[df["estado_civil"] == 0].sort_values("cliente_id", ascending=False).head(5))

estado_pago_mes_anomalo = [-2, 0]
for estado in estado_pago_mes_anomalo:
    for i in range(1, 7):
        format_section(
            title=f"Análisis de Variable estado de pago de mes {i} no documentada",
            subtitle=f"Estado de pago de mes {i} = {estado}"
        )

        display(df[df[f"estado_pago_mes_{i}"] == estado].sort_values("cliente_id", ascending=False).head(5))

for i in range(1, 7):
    format_section(
        title=f"Análisis de Variable monto facturado por mes {i}; variable negativa",
        subtitle=f"Monto facturado por mes {i} < 0"
    )

    display(df[df[f"monto_facturado_mes_{i}"] < 0].sort_values("cliente_id", ascending=False).head(5))

for i in range(1, 7):
    format_section(
        title=f"Análisis de variables monto facturado por mes {i} y pago del mes {i}",
        subtitle=f"Pago del mes {i} > Monto facturado por mes {i}"
    )

    display(df[df[f"pago_mes_{i}"] > df[f"monto_facturado_mes_{i}"]].sort_values("cliente_id", ascending=False).head(5))

format_section(
    title="Análisis de Valores extremos en limite de credito"
)
display(df[df["limite_credito"] >= df["limite_credito"].quantile(0.95)])

| Variable                             | Valor / condición detectada                | Interpretación                                                                            | Decisión de transformación                 | Justificación para ML                                                                                                                           |
| ------------------------------------ |--------------------------------------------| ----------------------------------------------------------------------------------------- | ------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------- |
| `educacion`                          | `0`                                        | Categoría no documentada / sin clasificación clara                                        | **Agrupar con `Otros/Desconocido`**        | Evita crear una categoría con muy pocas observaciones y reduce ruido en variables categóricas.                                                  |
| `educacion`                          | `4`, `5`, `6`                              | Categorías poco claras o residuales según la documentación disponible                     | **Agrupar con `Otros/Desconocido`**        | Las categorías con baja frecuencia pueden dificultar la generalización del modelo.                                                              |
| `estado_civil`                       | `0`                                        | Valor no documentado; probablemente representa información no disponible o no clasificada | **Agrupar con `Otros/Desconocido`**        | Se conserva el registro, pero se evita interpretar `0` como una categoría ordinal o con significado numérico.                                   |
| `estado_pago_mes_X`                  | `-2`                                       | Sin consumo / cuenta sin actividad                                                        | **Conservar como categoría independiente** | Es información relevante sobre el comportamiento financiero. No debe tratarse como "menos meses de atraso".                                     |
| `estado_pago_mes_X`                  | `-1`                                       | Pago puntual                                                                              | **Conservar como categoría independiente** | Representa un comportamiento financiero distinto de `0` y de los estados de mora.                                                               |
| `estado_pago_mes_X`                  | `0`                                        | Comportamiento normal asociado al pago mínimo / uso del crédito sin atraso                | **Conservar como categoría independiente** | Tiene significado semántico propio y no debe interpretarse simplemente como "0 meses de mora".                                                  |
| `estado_pago_mes_X`                  | `1` a `9`                                  | Retrasos de diferentes magnitudes                                                         | **Conservar como variable ordinal**        | Existe un orden natural: valores mayores representan mayor severidad del atraso. Puede ser una señal predictiva importante para `default_pago`. |
| `monto_facturado_mes_X`              | `< 0`                                      | Posible ajuste contable, devolución, saldo a favor u otra operación financiera            | **No eliminar automáticamente**            | Un valor negativo puede representar una situación financiera real. Eliminarlo sin evidencia podría destruir información predictiva.             |
| `pago_mes_X > monto_facturado_mes_X` | Pago superior a la facturación del período | Posible pago de deuda anterior, pago anticipado, liquidación de saldo o ajuste            | **Conservar**                              | No necesariamente es un error. Puede utilizarse para crear *features* como `pago/facturación` o diferencia entre pago y facturación.            |
| `limite_credito >= percentil 95`     | Valores extremos de límite de crédito      | Clientes con límites crediticios elevados                                                 | **Conservar**                              | El análisis no muestra evidencia suficiente de error. Ser un valor extremo estadísticamente no significa ser un dato incorrecto.                |
| Valores nulos                        | `0` casos                                  | No existen valores faltantes en el dataset actual                                         | **No realizar imputación**                 | No es necesario introducir valores artificiales mediante media, mediana o moda.                                                                 |
| Filas duplicadas                     | `0` casos                                  | No se detectaron registros completamente duplicados                                       | **No eliminar duplicados**                 | No existe evidencia de duplicación exacta que deba corregirse.                                                                                  |

In [ ]:
df_copia = df.copy()
df_copia["educacion"] = df_copia["educacion"].map(
    {1: "Posgrado", 2: "Universidad", 3: "Preparatoria", 4 : "Otros/Desconocido", 5 : "Otros/Desconocido", 0 : "Otros/Desconocido"})
df_copia["sexo"] = df_copia["sexo"].map({1: "Masculino", 2: "Femenino"})
df_copia["estado_civil"] = df_copia["estado_civil"].map({1: "Casado", 2: "Soltero", 3: "Otros/Desconocido", 0 : "Otros/Desconocido"})
df_copia["default_pago"] = df_copia["default_pago"].map({1: "si", 0: "No"})

for i in range(1, 7):
    df_copia[f"estado_pago_mes_{i}"] = df_copia[f"estado_pago_mes_{i}"].map(
         {
        -2: "Sin consumo",
        -1: "Pago puntual",
        0: "Pago mínimo",
        1: "retraso de 1 mes",
        2: "retraso de 2 meses",
        3: "retraso de 3 meses",
        4: "retraso de 4 meses",
        5: "retraso de 5 meses",
        6: "retraso de 6 meses",
        7: "retraso de 7 meses",
        8: "retraso de 8 meses",
        9: "retraso de 9 meses"
         }
    )

In [ ]:
conteo = df_copia["sexo"].value_counts()
porcentajes = (conteo / conteo.sum()) * 100

fig, ax = plt.subplots(figsize=(7, 5))

barras = ax.bar(conteo.index.astype(str), conteo.values, color="teal", edgecolor="black")

etiquetas_porcentaje = [f"{p:.1f}%" for p in porcentajes]

ax.bar_label(
    barras,
    labels=etiquetas_porcentaje,
    label_type="center",
    color="white",
    fontweight="bold",
    fontsize=11
)

ax.set_title("Distribución por Sexo", fontsize=12, pad=12)
ax.set_xlabel("Sexo", fontsize=10)
ax.set_ylabel("Cantidad de Clientes", fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
conteo = df_copia["educacion"].value_counts()
porcentajes = (conteo / conteo.sum()) * 100

fig, ax = plt.subplots(figsize=(8, 5))

barras = ax.bar(conteo.index.astype(str), conteo.values, color="teal", edgecolor="black")

etiquetas_porcentaje = [f"{p:.1f}%" for p in porcentajes]

ax.bar_label(
    barras,
    labels=etiquetas_porcentaje,
    label_type="center",
    color="white",
    fontweight="bold",
    fontsize=10
)

ax.set_title("Distribución por Nivel de Educación", fontsize=12, pad=12)
ax.set_xlabel("Nivel de Educación", fontsize=10)
ax.set_ylabel("Cantidad de Clientes", fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
conteo = df_copia["estado_civil"].value_counts()
porcentajes = (conteo / conteo.sum()) * 100

fig, ax = plt.subplots(figsize=(8, 5))

barras = ax.bar(conteo.index.astype(str), conteo.values, color="teal", edgecolor="black")

etiquetas_porcentaje = [f"{p:.1f}%" for p in porcentajes]

ax.bar_label(
    barras,
    labels=etiquetas_porcentaje,
    label_type="center",
    color="white",
    fontweight="bold",
    fontsize=10
)

ax.set_title("Distribución por Estado Civil", fontsize=12, pad=12)
ax.set_xlabel("Estado Civil", fontsize=10)
ax.set_ylabel("Cantidad de Clientes", fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
conteo = df_copia["default_pago"].value_counts()
porcentajes = (conteo / conteo.sum()) * 100

fig, ax = plt.subplots(figsize=(8, 5))

barras = ax.bar(conteo.index.astype(str), conteo.values, color="teal", edgecolor="black")

etiquetas_porcentaje = [f"{p:.1f}%" for p in porcentajes]

ax.bar_label(
    barras,
    labels=etiquetas_porcentaje,
    label_type="center",
    color="white",
    fontweight="bold",
    fontsize=10
)

ax.set_title("Distribución por Incumplimiento de Pago", fontsize=12, pad=12)
ax.set_xlabel("Estado de Incumplimiento de pago", fontsize=10)
ax.set_ylabel("Cantidad de Clientes", fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
cols_meses = [f"estado_pago_mes_{i}" for i in range(1, 7)]
df_meses = pd.DataFrame({f"Mes {i}": df_copia[col].value_counts() for i, col in enumerate(cols_meses, 1)}).fillna(0)

ax = df_meses.T.plot(kind="bar", stacked=True, figsize=(10, 6), colormap="Spectral")

ax.set_title("Evolución del Estado de Pago (Mes 1 al Mes 6)")
ax.set_xlabel("Mes de Seguimiento")
ax.set_ylabel("Frecuencia de Clientes")
plt.xticks(rotation=0)
plt.legend(title="Estado", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

plt.show()

In [ ]:
fig, ax = plt.subplots()

mora_grave = df_meses.loc[["retraso de 3 meses", "retraso de 4 meses",
                          "retraso de 5 meses", "retraso de 6 meses",
                          "retraso de 7 meses", "retraso de 8 meses"]]
mora_grave.T.plot(kind="bar", title="Detalle Mora Grave (3+ meses)", ax=ax, stacked=True)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

ax1.hist(df_copia["edad"], bins=15, color="skyblue", edgecolor="black", alpha=0.8)
ax1.set_title("Distribución de Edad (Histograma)", fontsize=12)
ax1.set_ylabel("Frecuencia (Clientes)")
ax1.grid(axis="y", linestyle="--", alpha=0.5)

sns.kdeplot(df_copia["edad"], ax=ax2, color="firebrick", linewidth=2, fill=True, alpha=0.2)
ax2.set_title("Estimación de Densidad de Kernel (KDE)", fontsize=12)
ax2.set_xlabel("Edad (años)")
ax2.set_ylabel("Densidad")
ax2.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.boxplot(
    df_copia["edad"],
    vert=False,
    patch_artist=True,
    notch=True,
    showmeans=True,
    meanline=True,
    showfliers=True,
    whis=1.5,

    boxprops=dict(facecolor="teal", color="black", linewidth=1.5),
    medianprops=dict(color="yellow", linewidth=2),
    meanprops=dict(color="red", linestyle="--", linewidth=2),
    flierprops=dict(marker="o", markerfacecolor="red", markersize=4, alpha=0.5),
    whiskerprops=dict(color="black", linestyle="--"),
    capprops=dict(color="black", linewidth=2)
)

ax.set_title("Distribucion de edad de los clientes")
ax.set_xlabel("Edad (años)")

plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True, sharey=True)
axes = axes.flatten()

for i in range(1, 7):
    ax = axes[i - 1]
    columna = f"monto_facturado_mes_{i}"

    ax.hist(df_copia[columna] / 1000, bins=40, log=True, color="teal", edgecolor="black")

    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${int(x):,}k'))

    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, pos: f'{int(y):,}'))

    ax.set_title(f"Mes {i}")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

# Etiquetas globales
fig.supxlabel("Monto Facturado (en Miles de $)", fontsize=12)
fig.supylabel("Cantidad de Clientes", fontsize=12)
fig.suptitle("Evolución del Monto Facturado (Mes 1 al Mes 6)", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
datos_meses = [df_copia[f"monto_facturado_mes_{i}"].dropna() / 1000 for i in range(1, 7)]
etiquetas = [f"Mes {i}" for i in range(1, 7)]

fig, ax = plt.subplots(figsize=(12, 6))

ax.boxplot(
    datos_meses,
    vert=False,
    tick_labels=etiquetas,
    patch_artist=True,
    showfliers=False,
    showmeans=True,
    meanline=True,

    boxprops=dict(facecolor="teal", color="white", linewidth=1.2),
    medianprops=dict(color="yellow", linewidth=2),
    meanprops=dict(color="red", linestyle="--", linewidth=1.5),
    whiskerprops=dict(color="white", linestyle="--"),
    capprops=dict(color="white", linewidth=1.5)
)

ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${int(x):,}k'))
ax.set_title("Evolución del Monto Facturado - Vista Enfocada (50% Central y Mediana)", fontsize=13, pad=15)
ax.set_xlabel("Monto Facturado (en Miles de $)", fontsize=11)
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
datos_meses = [df_copia[f"monto_facturado_mes_{i}"].dropna() / 1000 for i in range(1, 7)]
etiquetas = [f"Mes {i}" for i in range(1, 7)]

fig, ax = plt.subplots(figsize=(12, 6))

ax.boxplot(
    datos_meses,
    vert=False,
    tick_labels=etiquetas,
    patch_artist=True,
    showfliers=True,
    showmeans=True,
    meanline=True,

    boxprops=dict(facecolor="teal", color="white", linewidth=1.2),
    medianprops=dict(color="yellow", linewidth=2),
    meanprops=dict(color="red", linestyle="--", linewidth=1.5),
    whiskerprops=dict(color="white", linestyle="--"),
    capprops=dict(color="white", linewidth=1.5),

    flierprops=dict(marker="o", markerfacecolor="crimson", markeredgecolor="none", markersize=3, alpha=0.2)
)

ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${int(x):,}k'))
ax.set_title("Evolución del Monto Facturado - Vista Completa (Con Atípicos)", fontsize=13, pad=15)
ax.set_xlabel("Monto Facturado (en Miles de $)", fontsize=11)
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True, sharey=True)
axes = axes.flatten()

for i in range(1, 7):
    ax = axes[i - 1]
    columna = f"pago_mes_{i}"

    ax.hist(df_copia[columna] / 1000, bins=40, log=True, color="teal", edgecolor="black")

    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${int(x):,}k'))

    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, pos: f'{int(y):,}'))

    ax.set_title(f"Mes {i}")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

fig.supxlabel("Monto Pagado (en Miles de $)", fontsize=12)
fig.supylabel("Cantidad de Clientes", fontsize=12)
fig.suptitle("Evolución de los Pagos (Mes 1 al Mes 6)", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
datos_pagos = [df_copia[f"pago_mes_{i}"].dropna() / 1000 for i in range(1, 7)]
etiquetas = [f"Mes {i}" for i in range(1, 7)]

fig, ax = plt.subplots(figsize=(12, 6))

ax.boxplot(
    datos_pagos,
    vert=False,
    tick_labels=etiquetas,
    patch_artist=True,
    showfliers=False,
    showmeans=True,
    meanline=True,

    boxprops=dict(facecolor="teal", color="white", linewidth=1.2),
    medianprops=dict(color="yellow", linewidth=2),
    meanprops=dict(color="red", linestyle="--", linewidth=1.5),
    whiskerprops=dict(color="white", linestyle="--"),
    capprops=dict(color="white", linewidth=1.5)
)

ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${int(x):,}k'))
ax.set_title("Evolución de los Pagos - Vista Enfocada (50% Central y Mediana)", fontsize=13, pad=15)
ax.set_xlabel("Monto Pagado (en Miles de $)", fontsize=11)
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
datos_pagos = [df_copia[f"pago_mes_{i}"].dropna() / 1000 for i in range(1, 7)]
etiquetas = [f"Mes {i}" for i in range(1, 7)]

fig, ax = plt.subplots(figsize=(12, 6))

ax.boxplot(
    datos_pagos,
    vert=False,
    tick_labels=etiquetas,
    patch_artist=True,
    showfliers=True,
    showmeans=True,
    meanline=True,

    boxprops=dict(facecolor="teal", color="white", linewidth=1.2),
    medianprops=dict(color="yellow", linewidth=2),
    meanprops=dict(color="red", linestyle="--", linewidth=1.5),
    whiskerprops=dict(color="white", linestyle="--"),
    capprops=dict(color="white", linewidth=1.5),
    flierprops=dict(marker="o", markerfacecolor="crimson", markeredgecolor="none", markersize=3, alpha=0.2)
)

ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${int(x):,}k'))
ax.set_title("Evolución de los Pagos - Vista Completa (Con Atípicos)", fontsize=13, pad=15)
ax.set_xlabel("Monto Pagado (en Miles de $)", fontsize=11)
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

ax1.hist(
    df_copia["limite_credito"] / 1000,
    bins=25,
    color="teal",
    edgecolor="black",
    alpha=0.8,
)
ax1.set_title("Distribución del Límite de Crédito (Histograma)", fontsize=12)
ax1.set_ylabel("Frecuencia (Clientes)")
ax1.grid(axis="y", linestyle="--", alpha=0.5)

sns.kdeplot(
    df_copia["limite_credito"] / 1000,
    ax=ax2,
    color="firebrick",
    linewidth=2,
    fill=True,
    alpha=0.2,
)
ax2.set_title("Estimación de Densidad de Kernel (KDE)", fontsize=12)
ax2.set_xlabel("Límite de Crédito (en Miles de $)", fontsize=10)
ax2.set_ylabel("Densidad")
ax2.grid(axis="y", linestyle="--", alpha=0.5)

ax2.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: f"${int(x):,}k")
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.boxplot(
    df_copia["limite_credito"].dropna() / 1000,
    vert=False,
    tick_labels=["Límite Crédito"],
    patch_artist=True,
    notch=True,
    showmeans=True,
    meanline=True,
    showfliers=False,

    boxprops=dict(facecolor="teal", color="white", linewidth=1.2),
    medianprops=dict(color="yellow", linewidth=2),
    meanprops=dict(color="red", linestyle="--", linewidth=1.5),
    whiskerprops=dict(color="white", linestyle="--"),
    capprops=dict(color="white", linewidth=1.5)
)

ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${int(x):,}k'))
ax.set_title("Límite de Crédito - Vista Enfocada (50% Central y Mediana)", fontsize=12, pad=12)
ax.set_xlabel("Límite de Crédito (en Miles de $)", fontsize=10)
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.boxplot(
    df_copia["limite_credito"].dropna() / 1000,
    vert=False,
    tick_labels=["Límite Crédito"],
    patch_artist=True,
    notch=True,
    showmeans=True,
    meanline=True,
    showfliers=True,

    boxprops=dict(facecolor="teal", color="white", linewidth=1.2),
    medianprops=dict(color="yellow", linewidth=2),
    meanprops=dict(color="red", linestyle="--", linewidth=1.5),
    whiskerprops=dict(color="white", linestyle="--"),
    capprops=dict(color="white", linewidth=1.5),
    flierprops=dict(marker="o", markerfacecolor="crimson", markeredgecolor="none", markersize=3, alpha=0.2)
)

ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${int(x):,}k'))
ax.set_title("Límite de Crédito - Vista Completa (Con Atípicos)", fontsize=12, pad=12)
ax.set_xlabel("Límite de Crédito (en Miles de $)", fontsize=10)
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
matriz_corr = df.drop(columns=["cliente_id"]).corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(14, 11))

sns.heatmap(
    matriz_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=ax
)

ax.set_title("Matriz de Correlación Numérica", fontsize=14, pad=15)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
corr_default = (
    df.drop(columns=["cliente_id"])
    .corr(numeric_only=True)["default_pago"]
    .drop("default_pago")
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 6))

barras = ax.barh(corr_default.index, corr_default.values, color="teal", edgecolor="black")

ax.bar_label(barras, fmt="%.2f", padding=3, fontsize=9)

ax.axvline(0, color="white", linestyle="--", linewidth=1)

ax.set_title("Correlación de cada Variable con el Default de Pago", fontsize=12, pad=12)
ax.set_xlabel("Coeficiente de Correlación de Pearson", fontsize=10)
ax.grid(axis="x", linestyle="--", alpha=0.4)

ax.set_xlim(-0.2, corr_default.max() + 0.1)

plt.tight_layout()
plt.show()

In [ ]:
format_section(
    title="Análisis Agrupado: Estado de Pago Mes 6",
    subtitle="Promedio por categoría de cumplimiento",
)

df_res = df.groupby("default_pago")["estado_pago_mes_6"].mean().reset_index()
df_res["default_pago"] = df_res["default_pago"].map(
    {0: "Al día / Cumplió (0)", 1: "En Default / Incumplió (1)"}
)

df_res["estado_pago_mes_6"] = df_res["estado_pago_mes_6"].round(4)

headers = ["Estado de Default", "Promedio Estado Pago (Mes 6)"]
data_rows = df_res.values.tolist()

format_table(headers=headers, data=data_rows, render=True)

In [ ]:
grupo_no_default = df[df["default_pago"] == 0]["estado_pago_mes_6"].dropna()
grupo_default = df[df["default_pago"] == 1]["estado_pago_mes_6"].dropna()

datos = [grupo_no_default, grupo_default]
etiquetas = ["Al día (0)", "En Default (1)"]

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(
    datos,
    vert=True,
    tick_labels=etiquetas,
    patch_artist=True,
    showmeans=True,
    meanline=True,
    showfliers=True,

    boxprops=dict(facecolor="teal", color="white", linewidth=1.2),
    medianprops=dict(color="yellow", linewidth=2),
    meanprops=dict(color="red", linestyle="--", linewidth=1.5),
    whiskerprops=dict(color="white", linestyle="--"),
    capprops=dict(color="white", linewidth=1.5),

    flierprops=dict(marker="o", markerfacecolor="crimson", markeredgecolor="none", markersize=4, alpha=0.25)
)

ax.set_title("Estado de Pago (Mes 6) según Condición de Default", fontsize=13, pad=15)
ax.set_xlabel("Estado de Default de Pago", fontsize=10)
ax.set_ylabel("Estado de Pago en Mes 6 (Categoría/Meses de Mora)", fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()